In [4]:
import pandas as pd

In [5]:
df_clientes = pd.read_csv('naranerd_clientes.csv')
df_produtos = pd.read_csv('naranerd_produtos.csv')
df_vendedores = pd.read_csv('naranerd_vendedores.csv')
df_vendas = pd.read_csv('naranerd_vendas.csv')

In [ ]:
df_clientes[['nome','cidade']]

,id_cliente,nome,email,cidade
0,1,Carla Correia,carla.correia1@email.com,Nova Iguaçu
1,2,João Gomes,joão.gomes2@email.com,Cabo Frio
2,3,Bruno Rocha,bruno.rocha3@email.com,Cabo Frio
3,4,Karina Souza,karina.souza4@email.com,São Gonçalo
4,5,Bernardo Freitas,bernardo.freitas5@email.com,Nova Iguaçu
...,...,...,...,...
245,246,Ximena Souza,ximena.souza246@email.com,Volta Redonda
246,247,Gabriela Rocha,gabriela.rocha247@email.com,Rio de Janeiro
247,248,Camila Souza,camila.souza248@email.com,Nova Iguaçu
248,249,Yasmin Correia,yasmin.correia249@email.com,São João de Meriti


In [ ]:
df_produtos 


In [ ]:
df_vendas [['id_produto', 'valor']]

In [ ]:
df_vendedores.head()

In [ ]:
df_vendas.loc[df_vendas['valor'] > 100][['id_produto', 'valor']]

In [ ]:

df_vendas.loc[(df_vendas['id_cliente'] == 3) & (df_vendas['valor'] > 100)]

In [ ]:
qtd_loja = df_vendas.groupby('id_vendedor').size().reset_index(name= 'qtde vendas')
qtd_loja.loc[qtd_loja['qtde vendas'] > 350]
qtd_loja


In [ ]:
resultado = pd.merge(df_vendas, df_produtos,on='id_produto', how='inner')
resultado[['id_venda', 'nome', 'categoria', 'valor']]


In [ ]:
df_fat_cat = resultado.groupby('categoria')['valor'].sum().reset_index().sort_values(by='valor', ascending=False)

df_fat_cat.loc[df_fat_cat['valor'] > 60000]

In [ ]:
#1  Quem é esse cliente e quanto ele gastou?
SELECT c.id_cliente, c.nome, SUM(v.valor) AS total_gasto
FROM  vendas v
INNER JOIN clientes c  ON v.id_cliente = c.id_cliente
GROUP BY c.nome, c.id_cliente
ORDER BY total_gasto DESC

In [ ]:
# 1  Quem é esse cliente e quanto ele gastou?
df_clientes_vendas = pd.merge(df_vendas, df_clientes, on= 'id_cliente', how='inner')
df_gasto_cli = df_clientes_vendas.groupby(['id_cliente','nome'])['valor'].sum().reset_index(name='total_gasto').sort_values(by='total_gasto', ascending =False)

df_gasto_cli


In [ ]:
-- 2 ranking de vendedores.
SELECT r.nome, r.id_vendedor, COUNT(v.id_venda) AS quantidade_vendas
FROM vendas v
INNER JOIN vendedores r  ON v.id_vendedor = r.id_vendedor 
GROUP BY r.id_vendedor, r.nome
ORDER BY quantidade_vendas DESC

In [ ]:
#  ranking de vendedores.
df_vendas_vendedores = pd.merge(df_vendas, df_vendedores, on = 'id_vendedor', how = 'inner')
df_rank_vendedor =df_vendas_vendedores.groupby(['id_vendedor', 'nome'])['quantidade'].count().reset_index(name='qtde_vendas').sort_values(by='qtde_vendas', ascending = False)
df_rank_vendedor

In [ ]:
-- 3  categoria de produto trouxe mais receit
SELECT p.categoria, SUM(v.valor) AS categoria_receita
FROM vendas v
INNER JOIN produtos p 	ON  v.id_produto = p.id_produto 
GROUP BY p.categoria
ORDER BY categoria_receita DESC

In [ ]:
# 3  categoria de produto trouxe mais receit
df_vendas_produto = pd.merge(df_vendas, df_produtos, on='id_produto', how = 'inner')
df_rank_cat = df_vendas_produto.groupby(['categoria'])['valor'].sum().reset_index(name='cat_receita').sort_values(by='cat_receita', ascending=False)
df_rank_cat


In [ ]:
-- 4 vendedor cidade
SELECT r.nome, c.cidade, COUNT(v.id_venda) AS cidades_vendas
FROM vendas v
JOIN clientes c   ON  v.id_cliente = c.id_cliente 
JOIN vendedores r ON  v.id_vendedor = r.id_vendedor
GROUP BY c.cidade, r.nome
ORDER BY cidades_vendas DESC

In [117]:
#4 vendedor cidade  
df_vendas_clientes = pd.merge(df_vendas, df_clientes, on = 'id_cliente', how = 'inner')
df_vendas_vend = pd.merge(df_vendas_clientes, df_vendedores, on = 'id_vendedor', how = 'inner')
df_cidade_vendedor = df_vendas_vend.groupby(['cidade', 'nome_y'])['id_venda'].count().reset_index(name='vendedor_cidade').sort_values(by='vendedor_cidade', ascending = False)
df_cidade_vendedor

,cidade,nome_y,vendedor_cidade
117,São Gonçalo,Ricardo Dias,293
46,Niterói,Caio Araújo,286
98,Rio de Janeiro,Leonardo Almeida,266
127,São João de Meriti,Juliana Rocha,265
141,Volta Redonda,Juliana Almeida,260
...,...,...,...
35,Duque de Caxias,Fernanda Silva,5
7,Belford Roxo,Juliana Rocha,4
1,Belford Roxo,Caio Araújo,4
12,Belford Roxo,Ricardo Dias,3


In [ ]:
#4 vendedor cidade  
df_vendas_clientes = pd.merge(df_vendas, df_clientes, on = 'id_cliente', how = 'inner')
df_vendas_vend = pd.merge(df_vendas_clientes, df_vendedores, on = 'id_vendedor', how = 'inner', suffixes=('_cliente', '_vendedor'))
df_cidade_vendedor = df_vendas_vend.groupby(['cidade', 'nome_vendedor'])['id_venda'].count().reset_index(name='vendedor_cidade').sort_values(by='vendedor_cidade', ascending = False)
df_cidade_vendedor


In [ ]:
# 5 média venda
SELECT r.nome, p.categoria, ROUND(AVG(v.valor),2) AS ticket_médio
FROM vendas v
JOIN produtos p   ON  v.id_produto = p.id_produto 
JOIN vendedores r ON  v.id_vendedor = r.id_vendedor
GROUP BY r.nome, p.categoria
ORDER BY ticket_médio DESC

In [ ]:
# 5 média venda
df_med = pd.merge(df_vendas, df_produtos, on = 'id_produto', how ='inner')
df_med_ven = pd.merge(df_med, df_vendedores, on ='id_vendedor', how = 'inner')
df_med_vendedor = df_med_ven.groupby(['nome_y', 'categoria'])['valor'].mean().reset_index(name='méd_vendedor').sort_values(by='méd_vendedor', ascending=False)
df_med_vendedor

In [ ]:
# 5 média venda
df_med = pd.merge(df_vendas, df_produtos, on = 'id_produto', how ='inner')
df_med_ven = pd.merge(df_med, df_vendedores, on ='id_vendedor', how = 'inner')
df_med_ven = df_med_ven.rename(columns={'nome_y': 'nome_vendedor'})
df_med_vendedor = df_med_ven.groupby(['nome_vendedor', 'categoria'])['valor'].mean().reset_index(name='méd_vendedor').sort_values(by='méd_vendedor', ascending=False)
df_med_vendedor
